# Phase 1 — per-cell BOL parameter identification

Identify the electrochemical parameters PyBaMM needs to represent each of the 13 cohort cells at Beginning-of-Life:

| Parameter | From | How |
|---|---|---|
| `x_100, x_0, y_100, y_0` (stoichiometry endpoints) | OCV_SOC | Nonlinear LS anchoring Prada2013 half-cell OCPs to measured V(SoC) |
| `Q_n_Ah, Q_p_Ah` | Q_rpt + stoichiometry | `Q_n = Q_rpt/(x_100−x_0)`; `Q_p = Q_rpt/(y_0−y_100)` |
| `R0_Ohm` | HPPC | Median R0 across discharge pulses (extract.py) |
| `R1_Ohm` | HPPC | Median R1 across recovery segments (extract.py) |
| `D_s_m2_s` | GITT | `D_s = R_p² / τ_diff` with Prada2013 particle radius |

**Validation gate**: OCV RMSE over the upper-half plateau (SoC 0.50–0.90) must be < 20 mV. Cells that fail get logged but not silently skipped — we surface the failure.

**NOT identified here** (Phase 2 DE-fit territory): SEI + plating + LAM rates.

**Outputs**:
- One yaml per cell → `configs/bol_params/{make}_{cell}.yaml`
- Aggregate summary → `configs/bol_params/_aggregate.csv`
- Per-parameter cross-cohort plots at the bottom of this notebook

In [1]:
import importlib, sys, warnings, yaml
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
HERE     = Path('/home/hj/Desktop/PINNs/Voltaris/Data_Exploration')
CONFIGS  = Path('/home/hj/Desktop/PINNs/configs')
OUT_BOL  = CONFIGS / 'bol_params'; OUT_BOL.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(HERE))

import extract, phase1_bol
importlib.reload(extract); importlib.reload(phase1_bol)

manifest = yaml.safe_load((HERE / 'training_cohort.yaml').read_text())
COHORT   = {k: [str(c).zfill(4) for c in v] for k, v in manifest['cohort'].items()}
print(f'Cohort: {sum(len(v) for v in COHORT.values())} cells across {len(COHORT)} makes')
for m, cells in COHORT.items(): print(f'  {m}: {cells}')

NAMEPLATE = {'CALB': 72.0, 'EVE': 105.0, 'REPT': 150.0}

Cohort: 13 cells across 3 makes
  CALB: ['0003', '0008', '0009', '0010', '0015']
  EVE: ['0002', '0004', '0008']
  REPT: ['0004', '0007', '0012', '0046', '0057']


In [2]:
# Prada2013 half-cell OCPs — one call, reused for all 13 cells
U_p, U_n = phase1_bol.load_prada_ocps()

# Sanity check: a full LFP+graphite cell should give ~3.4 V at 50% SoC
soc = np.linspace(0.02, 0.98, 5)
print('Prada2013 half-cell OCPs — 5-point sanity table (soc: predicted V_cell)')
for s in soc:
    x = 0.122 + s * (0.879 - 0.122)
    y = 0.951 + s * (0.010 - 0.951)
    v = U_p(y)[0] - U_n(x)[0]
    print(f'  soc={s:.2f}   x_neg={x:.3f}  y_pos={y:.3f}  V_cell={v:.4f}')

Prada2013 half-cell OCPs — 5-point sanity table (soc: predicted V_cell)
  soc=0.02   x_neg=0.137  y_pos=0.932  V_cell=2.9854
  soc=0.26   x_neg=0.319  y_pos=0.706  V_cell=3.2403
  soc=0.50   x_neg=0.500  y_pos=0.480  V_cell=3.2649
  soc=0.74   x_neg=0.682  y_pos=0.255  V_cell=3.3100
  soc=0.98   x_neg=0.864  y_pos=0.029  V_cell=3.3217


In [3]:
# Load pre-computed RAW scalars (from extract.py)
sc = pd.read_csv(HERE / 'characterization_scalars.csv')
sc['cell_id'] = sc['cell_id'].astype(str).str.zfill(4)
sc = sc[sc.apply(lambda r: r['cell_id'] in COHORT.get(r['make'], []), axis=1)].copy()
sc = sc.set_index(['make','cell_id'])
print(f'Scalars loaded for {len(sc)} cohort cells')
display(sc[['capacity_Ah','HPPC_R0_mOhm','HPPC_R1_mOhm','GITT_tau_diff_s']].round(3))

Scalars loaded for 13 cohort cells


capacity_Ah  HPPC_R0_mOhm  HPPC_R1_mOhm  GITT_tau_diff_s
make cell_id                                                          
EVE  0002         104.270         1.717         0.347          813.901
CALB 0003          56.977         1.046         0.582          877.598
EVE  0004         103.810         0.794         0.353          843.914
REPT 0004         153.548         0.810         0.237          833.865
     0007         148.178         1.053         0.248          822.045
CALB 0008          50.898         1.370         0.769          876.297
EVE  0008         104.014         1.778         0.377          819.479
CALB 0009          53.775         1.122         0.638          871.775
     0010          52.571         1.332         0.724          866.809
REPT 0012         148.790         0.764         0.230          831.236
CALB 0015          56.049         1.047         0.597          879.384
REPT 0046         153.661         0.740         0.221          840.224
     0057         145.710         0.859         0.225          825.813

## 1. Fit stoichiometry + derive everything, per cell

One call to `phase1_bol.identify_cell` per cohort cell. Each call:
1. loads the cell's OCV curve via `extract.py::extract_ocv_curve`
2. runs L-BFGS-B on `(x_100, x_0, y_100, y_0)` bounded to physical ranges
3. derives `Q_n / Q_p` from the fitted stoichiometry + measured `Q_rpt`
4. computes `D_s` from `GITT_tau_diff_s`
5. copies `R0 / R1` from HPPC scalars
6. logs OCV RMSE (upper-half + full) for the validation gate

In [4]:
def _get(row_key, col):
    try: return float(sc.loc[row_key, col])
    except Exception: return np.nan

results: list[phase1_bol.CellBOL] = []
for make, cells in COHORT.items():
    for cell in cells:
        key = (make, cell)
        Q_rpt = _get(key, 'capacity_Ah')
        R0    = _get(key, 'HPPC_R0_mOhm')
        R1    = _get(key, 'HPPC_R1_mOhm')
        tau   = _get(key, 'GITT_tau_diff_s')
        bol = phase1_bol.identify_cell(
            make=make, cell=cell,
            Q_rpt_Ah=Q_rpt, HPPC_R0_mOhm=R0, HPPC_R1_mOhm=R1,
            GITT_tau_diff_s=tau,
            nominal_capacity_Ah=NAMEPLATE[make],
            U_p=U_p, U_n=U_n,
            extract_module=extract,
        )
        results.append(bol)
        st = bol.stoichiometry; v = bol.validation
        marker = '✓' if v.get('passed_upper_rmse_budget', False) else '✗'
        print(f"{marker} {make} {cell}  RMSE_upper={v.get('ocv_rmse_upper_half_mV', float('nan')):5.1f} mV  "
              f"x_100={st.get('x_100', float('nan')):.3f}  y_0={st.get('y_0', float('nan')):.3f}  "
              f"Q_n={bol.capacity.get('Q_n_Ah', float('nan')):5.1f} Ah  "
              f"D_s={bol.diffusion.get('D_s_m2_s', float('nan')):.2e} m²/s")

✓ CALB 0003  RMSE_upper=  3.6 mV  x_100=0.679  y_0=0.977  Q_n=104.1 Ah  D_s=3.10e-14 m²/s
✓ CALB 0008  RMSE_upper=  1.3 mV  x_100=0.655  y_0=0.600  Q_n= 88.7 Ah  D_s=3.11e-14 m²/s


✓ CALB 0009  RMSE_upper=  3.4 mV  x_100=0.664  y_0=0.978  Q_n=100.3 Ah  D_s=3.13e-14 m²/s
✓ CALB 0010  RMSE_upper=  3.4 mV  x_100=0.660  y_0=0.978  Q_n= 98.4 Ah  D_s=3.14e-14 m²/s


✓ CALB 0015  RMSE_upper=  4.4 mV  x_100=0.674  y_0=0.979  Q_n=104.0 Ah  D_s=3.10e-14 m²/s
✓ EVE 0002  RMSE_upper=  1.9 mV  x_100=0.878  y_0=0.950  Q_n=138.2 Ah  D_s=3.35e-14 m²/s


✓ EVE 0004  RMSE_upper=  1.3 mV  x_100=0.880  y_0=0.952  Q_n=138.0 Ah  D_s=3.23e-14 m²/s
✓ EVE 0008  RMSE_upper=  1.5 mV  x_100=0.880  y_0=0.949  Q_n=137.2 Ah  D_s=3.33e-14 m²/s


✓ REPT 0004  RMSE_upper=  2.4 mV  x_100=0.890  y_0=0.967  Q_n=201.4 Ah  D_s=3.27e-14 m²/s


✓ REPT 0007  RMSE_upper=  3.9 mV  x_100=0.879  y_0=0.972  Q_n=192.4 Ah  D_s=3.31e-14 m²/s


✓ REPT 0012  RMSE_upper=  2.6 mV  x_100=0.876  y_0=0.955  Q_n=198.7 Ah  D_s=3.28e-14 m²/s


✓ REPT 0046  RMSE_upper=  2.4 mV  x_100=0.894  y_0=0.965  Q_n=201.4 Ah  D_s=3.24e-14 m²/s


✓ REPT 0057  RMSE_upper=  2.0 mV  x_100=0.861  y_0=0.974  Q_n=196.8 Ah  D_s=3.30e-14 m²/s


## 2. Aggregate table across the cohort

In [5]:
def _flatten(r: phase1_bol.CellBOL) -> dict:
    return {
        'make': r.make, 'cell': r.cell,
        'nom_Ah': r.nominal_capacity_Ah,
        'x_100': r.stoichiometry.get('x_100'),
        'x_0':   r.stoichiometry.get('x_0'),
        'y_100': r.stoichiometry.get('y_100'),
        'y_0':   r.stoichiometry.get('y_0'),
        'Q_n':   r.capacity.get('Q_n_Ah'),
        'Q_p':   r.capacity.get('Q_p_Ah'),
        'Q_rpt': r.capacity.get('Q_rpt_used'),
        'R0_mΩ': (r.resistance.get('R0_Ohm') or np.nan) * 1000,
        'R1_mΩ': (r.resistance.get('R1_Ohm') or np.nan) * 1000,
        'D_s':   r.diffusion.get('D_s_m2_s'),
        'RMSE_up_mV':  r.validation.get('ocv_rmse_upper_half_mV'),
        'RMSE_full_mV':r.validation.get('ocv_rmse_full_mV'),
        'pass_gate':   r.validation.get('passed_upper_rmse_budget'),
        'bound_clip':  r.validation.get('stoich_bound_clipped'),
    }

agg = pd.DataFrame([_flatten(r) for r in results]).sort_values(['make','cell']).reset_index(drop=True)
display(Markdown('### Per-cell BOL parameters (aggregate)'))
display(agg.round({'x_100':3,'x_0':3,'y_100':4,'y_0':3,'Q_n':2,'Q_p':2,'Q_rpt':2,
                    'R0_mΩ':3,'R1_mΩ':3,'RMSE_up_mV':1,'RMSE_full_mV':1}))
agg.to_csv(OUT_BOL / '_aggregate.csv', index=False)
print(f'\nWrote: {OUT_BOL / "_aggregate.csv"}')

pass_rate = agg['pass_gate'].mean()
clip_rate = agg['bound_clip'].mean()
print(f'\nGate pass rate: {int(agg["pass_gate"].sum())}/{len(agg)} ({pass_rate:.0%})')
print(f'Bound-clipped:   {int(agg["bound_clip"].sum())}/{len(agg)} ({clip_rate:.0%})')

### Per-cell BOL parameters (aggregate)

,make,cell,nom_Ah,x_100,x_0,y_100,y_0,Q_n,Q_p,Q_rpt,R0_mΩ,R1_mΩ,D_s,RMSE_up_mV,RMSE_full_mV,pass_gate,bound_clip
0,CALB,0003,72.0,0.679,0.132,0.0102,0.977,104.10,58.94,56.98,1.046,0.582,3.104883e-14,3.6,5.0,True,False
1,CALB,0008,72.0,0.655,0.081,0.1000,0.600,88.73,101.80,50.90,1.370,0.769,3.109494e-14,1.3,14.5,True,True
2,CALB,0009,72.0,0.664,0.128,0.0131,0.978,100.25,55.72,53.78,1.122,0.638,3.125625e-14,3.4,4.7,True,False
3,CALB,0010,72.0,0.660,0.126,0.0160,0.978,98.41,54.66,52.57,1.332,0.724,3.143531e-14,3.4,4.8,True,False
4,CALB,0015,72.0,0.674,0.135,0.0117,0.979,104.01,57.93,56.05,1.047,0.597,3.098578e-14,4.4,5.4,True,False
5,EVE,0002,105.0,0.878,0.123,0.0170,0.950,138.18,111.74,104.27,1.717,0.347,3.347875e-14,1.9,6.7,True,False
6,EVE,0004,105.0,0.880,0.128,0.0092,0.952,138.02,110.08,103.81,0.794,0.353,3.228811e-14,1.3,6.9,True,False
7,EVE,0008,105.0,0.880,0.122,0.0173,0.949,137.22,111.58,104.01,1.778,0.377,3.325088e-14,1.5,6.6,True,False
8,REPT,0004,150.0,0.890,0.128,0.0138,0.967,201.39,161.09,153.55,0.810,0.237,3.267722e-14,2.4,12.5,True,False
9,REPT,0007,150.0,0.879,0.109,0.0314,0.972,192.45,157.58,148.18,1.053,0.248,3.314709e-14,3.9,24.2,True,False



Wrote: /home/hj/Desktop/PINNs/configs/bol_params/_aggregate.csv

Gate pass rate: 13/13 (100%)
Bound-clipped:   1/13 (8%)


## 3. Save per-cell yamls

In [6]:
for r in results:
    out = OUT_BOL / f'{r.make}_{r.cell}.yaml'
    out.write_text(yaml.safe_dump(r.to_dict(), sort_keys=False, allow_unicode=True))
print(f'Wrote {len(results)} per-cell yamls under {OUT_BOL}')
!ls -la {OUT_BOL}/ | head -20

Wrote 13 per-cell yamls under /home/hj/Desktop/PINNs/configs/bol_params
total 64
drwxrwxr-x 2 hj hj 4096 Jul 10 18:02 .
drwxrwxr-x 3 hj hj 4096 Jul 10 18:01 ..
-rw-rw-r-- 1 hj hj 3410 Jul 10 18:02 _aggregate.csv
-rw-rw-r-- 1 hj hj  889 Jul 10 18:02 CALB_0003.yaml
-rw-rw-r-- 1 hj hj  854 Jul 10 18:02 CALB_0008.yaml
-rw-rw-r-- 1 hj hj  887 Jul 10 18:02 CALB_0009.yaml
-rw-rw-r-- 1 hj hj  888 Jul 10 18:02 CALB_0010.yaml
-rw-rw-r-- 1 hj hj  889 Jul 10 18:02 CALB_0015.yaml
-rw-rw-r-- 1 hj hj  892 Jul 10 18:02 EVE_0002.yaml
-rw-rw-r-- 1 hj hj  890 Jul 10 18:02 EVE_0004.yaml
-rw-rw-r-- 1 hj hj  889 Jul 10 18:02 EVE_0008.yaml
-rw-rw-r-- 1 hj hj  892 Jul 10 18:02 REPT_0004.yaml
-rw-rw-r-- 1 hj hj  892 Jul 10 18:02 REPT_0007.yaml
-rw-rw-r-- 1 hj hj  890 Jul 10 18:02 REPT_0012.yaml
-rw-rw-r-- 1 hj hj  891 Jul 10 18:02 REPT_0046.yaml
-rw-rw-r-- 1 hj hj  890 Jul 10 18:02 REPT_0057.yaml


## 4. Per-parameter cross-cohort distributions

Same style as `cohort_parameter_distributions.ipynb` — one panel per BOL parameter, coloured by make. Look for cells that fall well outside the make's cluster (extraction / fit anomalies).

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

MAKE_COLOR = {'CALB': '#c94a3c', 'EVE': '#4dab5c', 'REPT': '#3c7cc9'}

PANELS = [
    ('x_100', 'x_100 (neg stoich charged)',   (0.60, 0.99)),
    ('x_0',   'x_0 (neg stoich discharged)',  (0.01, 0.30)),
    ('y_100', 'y_100 (pos stoich charged)',   (0.001, 0.10)),
    ('y_0',   'y_0 (pos stoich discharged)',  (0.60, 0.99)),
    ('Q_n',   'Q_n (Ah)',                     None),
    ('Q_p',   'Q_p (Ah)',                     None),
    ('R0_mΩ', 'HPPC R0 (mΩ)',                 (0.15, 5.0)),
    ('R1_mΩ', 'HPPC R1 (mΩ)',                 None),
    ('D_s',   'D_s (m²/s)',                   (1e-15, 1e-12)),
]
ncols = 3
nrows = int(np.ceil(len(PANELS) / ncols))
fig = make_subplots(rows=nrows, cols=ncols,
                     subplot_titles=[p[1] for p in PANELS],
                     vertical_spacing=0.09, horizontal_spacing=0.09)

y_pos = {'CALB': 0, 'EVE': 1, 'REPT': 2}

for i, (col, label, bounds) in enumerate(PANELS):
    row = i // ncols + 1; c = i % ncols + 1
    log_x = (col == 'D_s')
    for m in ['CALB','EVE','REPT']:
        sub = agg[agg['make'] == m]
        vals = sub[col].values.astype(float)
        cells = sub['cell'].values
        y_jitter = np.full(len(vals), y_pos[m], dtype=float) + \
                    np.random.default_rng(abs(hash(m+col)) % (2**32)).uniform(-0.15, 0.15, len(vals))
        fig.add_trace(go.Scatter(
            x=vals, y=y_jitter, mode='markers+text',
            marker=dict(size=10, color=MAKE_COLOR[m], line=dict(color='white', width=1.2)),
            text=cells, textposition='top center', textfont_size=8,
            showlegend=False,
            hovertemplate=(f'<b>{m} · %{{text}}</b><br>'
                            f'{label}: %{{x}}<extra></extra>'),
        ), row=row, col=c)
    if bounds is not None:
        lo, hi = bounds
        for x_ref in (lo, hi):
            if np.isfinite(x_ref):
                fig.add_vline(x=x_ref, row=row, col=c,
                               line=dict(color='grey', width=0.8, dash='dash'))
    fig.update_yaxes(tickmode='array',
                      tickvals=list(y_pos.values()),
                      ticktext=list(y_pos.keys()),
                      range=[-0.6, 2.6],
                      gridcolor='rgba(0,0,0,0.05)', row=row, col=c)
    fig.update_xaxes(gridcolor='rgba(0,0,0,0.08)', row=row, col=c,
                     type='log' if log_x else 'linear')

fig.update_layout(
    title=dict(text='Phase 1 BOL parameters — cross-cohort distribution (13 cells)',
                x=0.02, font_size=13),
    height=280*nrows+60, width=1150,
    plot_bgcolor='white',
    margin=dict(l=60, r=30, t=80, b=40),
)
fig.show()

## 5. What to look at

- **`pass_gate` column** in the aggregate table — any `False` means the OCV RMSE upper-half was ≥ 20 mV. That cell's stoichiometry fit doesn't match the observed OCV well enough. Options: relax bounds, weight the loss differently, or defer that cell.
- **`bound_clip` column** — `True` means at least one stoichiometry parameter hit its optimizer bound. That's a red flag that the fit is *forced* by the constraint, not by the data. Widen the bound or investigate the cell's OCV curve.
- **`Q_n vs Q_p`** — for LFP + graphite, these should be within a few percent of each other for a balanced cell. If they diverge markedly, either the stoichiometry fit put the operating window in the wrong place, or the RPT capacity isn't representative.
- **`D_s` cross-cohort** — should span 10⁻¹⁵ to 10⁻¹² m²/s (LFP literature range). Values outside are extraction artefacts.

**Ready for Phase 2** once the aggregate table is inspected and any outliers are either accepted or dropped. Each cohort cell's `configs/bol_params/{make}_{cell}.yaml` is what Phase 2 (per-cell DE fit of SEI + plating + LAM) will read.